### 0. 환경 설정 — 위젯/진행바 비활성화
VS Code Jupyter 에서 `load_dataset`·`from_pretrained` 가 띄우는 **ipywidget 진행바**가
`i is not a function` 렌더 오류를 내는 것을 막는다. **모든 import 보다 먼저** 실행해야
효과가 있으므로 노트북 최상단에 둔다. (전역 차단은 `~/.ipython` startup 에도 설정돼 있음)

In [ ]:
# ── widget 렌더러 오류 방지 (VS Code Jupyter) ─────────────────────
# tqdm.auto / HuggingFace 내부 progress bar가 ipywidget을 쓰면
# 'Error rendering output ... i is not a function'으로 깨질 수 있음.
# 모든 import보다 먼저 실행되어야 효과가 있으므로 노트북 최상단에 둔다.
import os
os.environ['TQDM_NOTEBOOK'] = 'false'              # tqdm: ipywidget 대신 텍스트 모드
os.environ['TOKENIZERS_PARALLELISM'] = 'false'     # tokenizer 경고 억제
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # HF Hub 다운로드 진행바 비활성

# datasets의 map/download 진행바도 위젯 대신 비활성화
try:
    from datasets.utils.logging import disable_progress_bar
    disable_progress_bar()
except Exception:
    pass

# transformers 내부 progress bar도 비활성화
try:
    from transformers.utils import logging as _hf_logging
    _hf_logging.disable_progress_bar()
except Exception:
    pass


### 1. 모델 & 프로세서 로드 + 추론용 설정
사전학습 Donut(`naver-clova-ix/donut-base`) 모델과 `DonutProcessor`(이미지 전처리 + 토크나이저)를 불러온다.

**중요 포인트**
- `model.loss_type = "ForCausalLM"` — VED 모델은 클래스명이 매핑과 안 맞아 `loss_type=None` 경고가 뜨는데, 기본 동작과 동일한 손실을 명시해 경고만 제거(동작 변화 없음).
- `image_processor.size` 는 **dict** `{"height":..,"width":..}` 형식이어야 한다(신버전). 입력 해상도를 `[960,720]` 로 낮춰 VRAM 절약.
- `decoder_start_token_id` = 디코더가 생성을 시작할 첫 토큰(`<s>`), `pad_token_id` 도 함께 지정.

In [ ]:
import torch
from transformers import DonutProcessor,VisionEncoderDecoderModel


model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base")

# VisionEncoderDecoderModel은 클래스명이 LOSS_MAPPING과 매칭 안 돼 loss_type이
# None으로 잡힘 → "loss_type=None ... unrecognized" 경고. 기본 fallback과 동일한
# 인과적 LM 손실을 명시해 경고 제거 (동작 변화 없음, ForCausalLMLoss).
model.loss_type = "ForCausalLM"

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Load processor
processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base")

# set decoder start token id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(['<s>'])[0]
model.config.pad_token_id = processor.tokenizer.pad_token_id

max_length = 512
image_size = [960, 720]

# update image_size of the encoder

# model.config.encoder.image_size = list(processor.image_processor.size.values()) # (height, width)
model.config.encoder.image_size = image_size # (height, width)
model.config.decoder.max_length = max_length

# resize processor and model to match
# 신버전 image_processor.size 는 dict 형식 {"height":..,"width":..} 를 기대 (리스트 X)
processor.image_processor.size = {"height": image_size[0], "width": image_size[1]}
processor.image_processor.do_align_long_axis = False

## Random sample

### (데모) 임의 이미지 1장 로드
아래 몇 셀은 **모델이 일단 도는지** 빠르게 확인하는 용도다. SROIE 가 아니라 `cats-image`
데이터의 고양이 사진을 쓰므로 결과 자체에 의미는 없고, **forward 가 에러 없이 loss 를
내는지**만 본다. 실제 영수증 파이프라인은 아래 "SROIE example" 부터 시작한다.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("huggingface/cats-image")
image = dataset["test"]["image"][0]

### 이미지 → pixel_values 전처리
`image_processor` 가 이미지를 설정 해상도로 리사이즈하고 픽셀을 정규화해 모델 입력
텐서(`pixel_values`)로 만든다.

In [ ]:
pixel_values = processor.image_processor(image, return_tensors="pt").pixel_values.to(device)


labels = processor.tokenizer(
    "an image of two cats chilling on a couch",
    return_tensors="pt",
).input_ids.to(device)


### forward 로 loss 계산 (Teacher Forcing)
`labels` 를 함께 넘기면 모델이 **디코더 입력을 자동 생성**(teacher forcing)하고
교차엔트로피 loss 를 반환한다. 여기선 고양이 캡션이라 값은 무의미하고, **학습 메커니즘이
동작하는지** 확인하는 용도다.

In [ ]:
# the forward function automatically creates the correct decoder_input_ids
loss = model(pixel_values=pixel_values, labels=labels).loss

loss 값(스칼라 텐서) 출력.

In [ ]:
loss

# SROIE example

### 2. SROIE 데이터셋 내려받기
실제 영수증 데이터셋(ICDAR-2019-SROIE)을 git clone 해 `data/` 로 복사한 뒤 임시 폴더와
박스 라벨(`data/box`)을 정리한다. 결과: `data/img`(영수증 이미지) + `data/key`(정답 JSON).
> ⚠️ `%%bash` 는 **노트북 커널의 현재 작업 디렉터리**에 clone 한다(여기선 `SROIE_donut/`).

In [ ]:
%%bash 
# clone repository
git clone https://github.com/zzzDavid/ICDAR-2019-SROIE.git
# copy data
cp -r ICDAR-2019-SROIE/data ./
# clean up
rm -rf ICDAR-2019-SROIE
rm -rf data/box

### 정답 JSON → `metadata.jsonl` 변환 (imagefolder 포맷)
`data/key/*.json` 의 각 정답을 **한 줄 JSON 문자열("text" 컬럼)** 로 만들어
`data/img/metadata.jsonl` 로 저장한다. 이렇게 두면 HuggingFace `imagefolder` 로더가
이미지와 정답 텍스트를 자동으로 짝지어 읽는다. 변환 후 원본 `key/` 폴더는 삭제.

In [ ]:
import os
import json
from pathlib import Path
import shutil

# define paths
base_path = Path("data")
metadata_path = base_path.joinpath("key")
image_path = base_path.joinpath("img")
# define metadata list
metadata_list = []

# parse metadata
for file_name in metadata_path.glob("*.json"):
  with open(file_name, "r") as json_file:
    # load json file
    data = json.load(json_file)
    # create "text" column with json string
    text = json.dumps(data)
    # add to metadata list if image exists
    if image_path.joinpath(f"{file_name.stem}.jpg").is_file():    
      metadata_list.append({"text":text,"file_name":f"{file_name.stem}.jpg"})
      # delete json file
      
# write jsonline file
with open(image_path.joinpath('metadata.jsonl'), 'w') as outfile:
    for entry in metadata_list:
        json.dump(entry, outfile)
        outfile.write('\n')

# remove old meta data
shutil.rmtree(metadata_path)

### `imagefolder` 로 데이터셋 로드
`data/img/`(이미지 + metadata.jsonl)를 HuggingFace Dataset 으로 로드한다.
`image` 컬럼(영수증 이미지)과 `text` 컬럼(정답 JSON 문자열)이 생긴다.

In [ ]:
from pathlib import Path
from datasets import load_dataset

# define paths
base_path = Path("data")
metadata_path = base_path.joinpath("key")
image_path = base_path.joinpath("img")
# Load dataset
dataset = load_dataset("imagefolder", data_dir=image_path, split="train")

print(f"Dataset has {len(dataset)} images")
print(f"Dataset features are: {dataset.features.keys()}")

### 첫 샘플 확인
데이터셋 0번 샘플의 이미지/정답 텍스트를 꺼내 구조를 눈으로 확인한다.

In [ ]:
image = dataset["image"][0]
text = dataset["text"][0]

### SROIE 필드 토큰 등록 + 임베딩 확장
정답에 등장하는 필드(`<s_total>`, `<s_date>`, `<s_company>`, `<s_address>` …)와 `<s>`/`</s>` 를
**special token 으로 추가**하고, 토크나이저 크기에 맞춰 디코더 임베딩 테이블을 늘린다.

**중요 포인트**
- 필드 토큰을 등록하지 않으면 디코더가 그 토큰을 **생성할 수 없다**(필수 단계).
- `mean_resizing=False` — 새 토큰을 기존 분포 기반이 아닌 **랜덤 초기화**(원본 Donut 레시피와 동일, 결정적).

In [ ]:
processor.tokenizer.add_special_tokens({"additional_special_tokens": ['<s_total>', '</s_total>', '<s_date>', '</s_date>', '<s_company>', '</s_company>', '<s_address>', '</s_address>', '<s>', '</s>']})
model.decoder.resize_token_embeddings(len(processor.tokenizer), mean_resizing=False)

### 평가용 샘플 + 정답 시퀀스 지정
45번 영수증 이미지와, 그 정답을 **Donut 토큰 형식**(`<s><s_total>$6.90</s_total>...</s>`)으로
직접 적은 문자열을 준비한다. 이 토큰 문자열이 곧 디코더가 맞춰야 할 타깃이다.

In [ ]:
image = dataset["image"][45]
text = "<s><s_total>$6.90</s_total><s_date>27 MAR 2018</s_date><s_company>UNIHAKKA INTERNATIONAL SDN BHD</s_company><s_address>12, JALAN TAMPOI 7/4,KAWASAN PARINDUSTRIAN TAMPOI,81200 JOHOR BAHRU,JOHOR</s_address></s>"

### 전처리 + 라벨 토큰화
이미지는 `pixel_values` 로, 정답 토큰 문자열은 `input_ids` 로 변환한다.
`labels` 는 `input_ids` 복사본이며, **패딩 위치를 `-100` 으로 마스킹**해 loss 계산에서 제외한다
(`CrossEntropyLoss` 는 `-100` 을 무시).

In [ ]:
pixel_values = processor.image_processor(image, return_tensors="pt").pixel_values.to(device)


input_ids = processor.tokenizer(
    text,
    return_tensors="pt",
    add_special_tokens=False,
    max_length=512,
    padding="max_length",
).input_ids.to(device)

labels = input_ids.clone()
labels[labels == processor.tokenizer.pad_token_id] = -100  # model doesn't need to predict pad token
labels = labels.to(device)

입력/라벨 텐서의 shape 확인(전처리가 의도대로 됐는지 점검).

In [ ]:
print(f"Pixel values: {pixel_values.shape}")
print(f"Labels: {labels.shape}")

`labels` 텐서 내용 확인(앞부분은 토큰 ID, 패딩은 `-100`).

In [ ]:
labels

### 라벨 역디코딩으로 정답 복원 확인
`-100`(패딩)을 제외한 라벨 토큰을 다시 문자열로 디코딩해, **원래 정답 시퀀스가 그대로
복원되는지** 검증한다. 토큰화/마스킹이 올바른지 확인하는 핵심 체크.

In [ ]:
processor.tokenizer.decode(labels[0][labels[0]!=-100])

### 실제 SROIE 샘플로 loss 계산
이번엔 의미 있는 영수증 샘플로 forward 를 돌려 loss 를 확인한다. 값이 정상 범위면
학습 입력 파이프라인(이미지+라벨)이 모델과 잘 맞물린다는 뜻이다.

In [ ]:
# the forward function automatically creates the correct decoder_input_ids
loss = model(pixel_values=pixel_values, labels=labels).loss

loss